In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Leaky integrate-and-fire neuron network (sparse connection list version)
A, B, C, I = 0, 1, 2, 3
N = 4
threshold = 256
pipe_shift = 6
# connections[sender] = list of (receiver, weight)
connections = [
    [(B, 300)],    # A -> B
    [(C, 300)],    # B -> C
    [],            # C -> nobody
    [(B, -400)]    # I -> B (inhibitory)
]

# poke_connections[poke] = list of (receiver, weight)
poke_connections = [
    [(A, 300)],    # poke 0 (row[0]) -> A
    [(I, 300)]     # poke 1 (row[1]) -> I
]

pokes = [[1,0]
         ,[0,0]
         ,[0,1]
         ,[1,0]
         ,[0,0]
         ,[0,0]
         ,[1,1]
         ,[1,0]
         ,[1,0]
         ,[0,0]]

pipes = []
voltages = [0] * N
last_spikes = [0] * N

inputs_saved = []
spikes_saved = []
va_saved = []
vb_saved = []
vc_saved = []
vinhib_saved = []


def neuron_model(total, v):
  spike = 0
  v = v - (v >> 3) + total
  if v >= threshold:      # fire when voltage crosses threshold
    v = 0                 # reset after spiking
    spike = 1
  return v, spike


for step, row in enumerate(pokes):
  spikes = [0] * N
  totals = [0] * N

  # deliver last step's neuron spikes
  for i in range(N):
    if last_spikes[i] == 1:
      for target, weight in connections[i]:
        totals[target] += weight

  # deliver this step's pokes
  for p in range(len(poke_connections)):
    if row[p] == 1:
      for target, weight in poke_connections[p]:
        totals[target] += weight

  # pipes: flow from start-of-step voltages, written into totals
  for a, b, size in pipes:
    flow = ((voltages[a] - voltages[b]) * size) >> pipe_shift
    totals[a] -= flow
    totals[b] += flow

  # update every neuron
  for t in range(N):
    voltages[t], spikes[t] = neuron_model(totals[t], voltages[t])

  # save outputs
  inputs_saved.append(row[0]*2 + row[1])
  spikes_saved.append(spikes[A]*8 + spikes[B]*4 + spikes[C]*2 + spikes[I])
  va_saved.append(voltages[A])
  vb_saved.append(voltages[B])
  vc_saved.append(voltages[C])
  vinhib_saved.append(voltages[I])

  last_spikes = spikes

  print(f"step {step:3d}  poke {row} spike_a {spikes[A]:3d} spike_b {spikes[B]:3d} spike_c {spikes[C]:3d} spike_inhib {spikes[I]:3d}")

np.savetxt("chain_inputs.txt", inputs_saved, fmt="%x")
np.savetxt("chain_expected_spikes.txt", spikes_saved, fmt="%x")
np.savetxt("chain_expected_va.txt", [v & 0xFFFF for v in va_saved], fmt="%04x")
np.savetxt("chain_expected_vb.txt", [v & 0xFFFF for v in vb_saved], fmt="%04x")
np.savetxt("chain_expected_vc.txt", [v & 0xFFFF for v in vc_saved], fmt="%04x")
np.savetxt("chain_expected_vi.txt", [v & 0xFFFF for v in vinhib_saved], fmt="%04x")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')